# Batcher: Generating Batches for Model Training

The `Batcher` class (`crest/data/batching/Batcher.py`) generates mini-batches
efficiently using background processes or threads. It works with both:

- **StructuredDataset** — in-memory arrays aligned on the sample dimension
  (`crest/data/loading/StructuredDataset.py`)
- **Dataset** — heterogeneous grids loaded lazily from disk (Zarr/NetCDF/TileDB)

This notebook covers:

1. **StructuredDataset** — creating a dataset from in-memory numpy arrays
2. **Basic iteration** — construction, looping, and shapes
3. **The features parameter** — controlling how batches are grouped
4. **Context manager and lifecycle** — `with`, `close()`, and `active_workers`
5. **repeat and shuffle** — epoch behaviour
6. **Integration with a CREST Model** — `fit`, `evaluate`, and `predict`

In [ ]:
# Allow crest to be imported from the examples/ sub-directory
%cd -q ..

In [ ]:
import numpy as np
from tensorflow.keras.layers import Dense
from tensorflow import TensorSpec

from crest import Node
from crest.data.loading import StructuredDataset
from crest.data import Batcher
from crest.model.Model import Model

## 1. StructuredDataset

`StructuredDataset` (`crest/data/loading/StructuredDataset.py`) is the lightweight
dataset class for data that fits in memory. It requires all arrays to be aligned on
their first (sample) dimension. Pass each array as a positional argument, and
optionally provide `labels` to name them.

Below we create a simple synthetic regression dataset: $y = 2x + \varepsilon$ with
500 samples.

In [ ]:
rng = np.random.default_rng(0)
n   = 500

x_data = rng.uniform(0, 1, size=(n, 1)).astype("float32")
y_data = (2 * x_data + 0.1 * rng.standard_normal((n, 1))).astype("float32")

ds = StructuredDataset(x_data, y_data, labels=["x", "y"])

print(f"labels : {ds.labels}")
print(f"x shape: {ds.data[0].shape}  dtype={ds.data[0].dtype}")
print(f"y shape: {ds.data[1].shape}  dtype={ds.data[1].dtype}")

## 2. Basic Iteration

Construct a `Batcher` with a dataset and a `batch_size`. The key parameters are:

| Parameter | Default | Description |
|---|---|---|
| `batch_size` | — | Samples per batch |
| `features` | `None` | Which features to extract and how to group them (see §3) |
| `workers` | `2` | Background processes for parallel batch generation |
| `repeat` | `False` | Whether to loop indefinitely after exhausting the dataset |
| `shuffle` | `True` | Whether to randomize sample order |

Setting `workers=0` generates batches in the main thread — no subprocess overhead,
which is ideal for small datasets and debugging.

In [ ]:
# repeat=False exhausts the dataset in one pass; shuffle=False preserves order
batcher = Batcher(ds, batch_size=100,
                  features=[["x"], ["y"]],
                  workers=0, repeat=False, shuffle=False)

for i, batch in enumerate(batcher):
    x_batch, y_batch = batch
    print(f"Batch {i}: x={x_batch['x'].shape}  y={y_batch['y'].shape}")

batcher.close()


## 3. The `features` Parameter

The `features` parameter controls both *which* features to include and *how* they
are grouped in each batch:

- **features=None** (default) — each batch is a list of `Sample` objects
  (used when working with heterogeneous `Dataset` inputs requiring window logic).
- **features=[]** — each batch is a single `dict` containing **all** features.
- **features=['a', 'b']** — each batch is a single `dict` containing only `'a'`
  and `'b'` (useful for prediction-only batchers).
- **features=[['a', 'b'], ['c']]** — each batch is a **tuple** of dicts, one per
  inner list. This is the standard training format: `(inputs_dict, targets_dict)`.

The nested structure can be arbitrarily deep; inner lists group features into
separate dicts, while flat lists combine them into one.


In [ ]:

# --- features=[['x'], ['y']] → list of two dicts (standard training format) ---
with Batcher(ds, batch_size=50, features=[["x"], ["y"]], workers=0) as b:
    batch = next(b)
x_dict, y_dict = batch
print("features=[['x'], ['y']] :")
print(f"  type(batch)      = {type(batch).__name__}  len={len(batch)}")
print(f"  x_dict keys      = {list(x_dict.keys())}  shape={x_dict['x'].shape}")
print(f"  y_dict keys      = {list(y_dict.keys())}  shape={y_dict['y'].shape}")

# --- features=['x'] → single dict with only 'x' (prediction format) ---
with Batcher(ds, batch_size=50, features=["x"], workers=0) as b:
    x_only = next(b)
print("\nfeatures=['x'] :")
print(f"  type(batch)      = {type(x_only).__name__}")
print(f"  keys={list(x_only.keys())}  shape={x_only['x'].shape}")

# --- features=[] → single dict with all features ---
with Batcher(ds, batch_size=50, features=[], workers=0) as b:
    all_feats = next(b)
print("\nfeatures=[] (all features) :")
print(f"  type(batch)      = {type(all_feats).__name__}")
print(f"  keys={list(all_feats.keys())}")


## 4. Context Manager and Lifecycle

`Batcher` implements the context manager protocol — `__enter__` starts any
background workers, `__exit__` calls `close()` which stops them and releases
resources. Using `with` is the preferred pattern because it guarantees cleanup
even if an exception occurs.

Without `with`, background processes keep running between batches. Always call
`batcher.close()` explicitly if not using the context manager.

`active_workers` returns the list of currently running background processes; it
is empty when `workers=0` or after `close()` is called.

In [ ]:
batcher = Batcher(ds, batch_size=100, features=[["x"], ["y"]], workers=0)

print("Before start  — active_workers:", batcher.active_workers)

with batcher as b:
    x, y = next(b)
    print("Inside 'with' — active_workers:", batcher.active_workers)

print("After 'with'  — active_workers:", batcher.active_workers)
print(f"Batch shapes: x={x['x'].shape}  y={y['y'].shape}")

## 5. `repeat` and `shuffle`

- **repeat=False** (default) — the batcher exhausts after one pass through all
  samples. The total number of batches is `ceil(n_samples / batch_size)`.
- **repeat=True** — yields batches indefinitely; always pair with a stopping
  condition (`steps_per_epoch` in `model.fit`, or a manual counter).
- **shuffle=True** (default) — samples are randomized within each epoch.

In [ ]:
# repeat=False: exhausts after one pass
count = 0
with Batcher(ds, batch_size=100, features=[["x"], ["y"]],
             workers=0, repeat=False, shuffle=False) as b:
    for _ in b:
        count += 1
print(f"repeat=False: {count} batches from {n} samples (batch_size=100)")

# repeat=True: continues indefinitely — stop after 8 batches (> one full epoch)
count = 0
with Batcher(ds, batch_size=100, features=[["x"], ["y"]],
             workers=0, repeat=True, shuffle=True) as b:
    for _ in b:
        count += 1
        if count >= 8:
            break
print(f"repeat=True:  {count} batches consumed before break (would run indefinitely)")

## 6. Integration with a CREST Model

A `Batcher` plugs directly into `Model.fit`, `Model.evaluate`, and
`Model.predict` (`crest/model/Model.py`). Each method opens and closes the
batcher's context manager internally, so no explicit `with` block is needed.

**Note for model.evaluate:** always pass `steps` so Keras knows when to stop.
Without it, Keras's `EpochIterator` cannot infer the iterator length and emits a
spurious `UserWarning` when `StopIteration` fires. Setting
`steps = len(dataset) // batch_size` is the standard convention.

**Note for model.predict:** close the batcher after predicting, since
`Model.predict` does not open its own context manager for non-Batcher inputs.

In [ ]:
# Build a simple one-layer regression model with CREST
dense = Dense(1)
htg = Node(
    node=lambda X: {"y": dense(X["x"])},
    name="linear",
    inputs={"x": TensorSpec(shape=[None, 1])},
    outputs={"y": TensorSpec(shape=[None, 1])},
)
model = Model(htg)
model.compile(optimizer="adam", loss="mse")

batch_size = 100
n_steps    = n // batch_size  # 500 // 100 = 5

# model.fit manages the Batcher context internally
batcher_train = Batcher(ds, batch_size=batch_size,
                        features=[["x"], ["y"]],
                        workers=0, repeat=True, shuffle=True)
model.fit(batcher_train, steps_per_epoch=n_steps, epochs=10, verbose=0)
batcher_train.close()
print("Training complete.")

In [ ]:
# model.evaluate: pass steps so Keras knows when to stop
batcher_eval = Batcher(ds, batch_size=batch_size,
                       features=[["x"], ["y"]],
                       workers=0, repeat=False, shuffle=False)
results = model.evaluate(batcher_eval, steps=n_steps, verbose=0, return_dict=True)
print("Evaluate:", results)
batcher_eval.close()

# model.predict: use features=["x"] (no targets needed)
batcher_pred = Batcher(ds, batch_size=batch_size,
                       features=["x"],
                       workers=0, repeat=False, shuffle=False)
preds = model.predict(batcher_pred)
batcher_pred.close()
print(f"Predict keys={list(preds.keys())}  shape={preds['y'].shape}")